In [0]:
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    countDistinct,
    first,
    round as spark_round
)

silver_table = "online_retail.silver.transactions_clean"
gold_table = "online_retail.gold.product_sales_summary"

silver_df = spark.table(silver_table)

print(f"Silver rows available: {silver_df.count():,}")

Silver rows available: 44,722


In [0]:
# Select records that qualify as positive sales

positive_sales_df = silver_df.filter(
    col("is_positive_sale")
)

positive_sales_row_count = positive_sales_df.count()

print(
    f"Positive-sale rows: "
    f"{positive_sales_row_count:,}"
)

Positive-sale rows: 43,453


In [0]:
# Aggregate to one row per batch and stock code

gold_df = (
    positive_sales_df
    .groupBy(
        "batch_id",
        "stock_code"
    )
    .agg(
        first(
            "description",
            ignorenulls=True
        ).alias("description"),

        spark_sum(
            col("quantity")
        ).alias("total_quantity_sold"),

        spark_round(
            spark_sum(col("line_total")),
            2
        ).alias("total_revenue"),

        countDistinct(
            col("invoice")
        ).alias("distinct_invoice_count"),

        countDistinct(
            col("customer_id")
        ).alias("distinct_customer_count")
    )
)

In [0]:
# Validate the Gold aggregation

gold_row_count = gold_df.count()

print(f"Gold DataFrame rows: {gold_row_count:,}")

top_products_df = (
    gold_df
    .orderBy(
        col("total_revenue").desc()
    )
    .limit(10)
)


top_products_df.show(10, truncate=False)

Gold DataFrame rows: 3,057
+--------+----------+----------------------------------+-------------------+-------------+----------------------+-----------------------+
|batch_id|stock_code|description                       |total_quantity_sold|total_revenue|distinct_invoice_count|distinct_customer_count|
+--------+----------+----------------------------------+-------------------+-------------+----------------------+-----------------------+
|2009-12 |DOT       |DOTCOM POSTAGE                    |49                 |18574.58     |49                    |0                      |
|2009-12 |85123A    |WHITE HANGING HEART T-LIGHT HOLDER|6406               |17255.35     |304                   |220                    |
|2009-12 |22086     |PAPER CHAIN KIT 50'S CHRISTMAS    |3362               |10169.36     |184                   |111                    |
|2009-12 |15056BL   |EDWARDIAN PARASOL BLACK           |2188               |8671.75      |69                    |48                     |
|2009-1

In [0]:
# Write the managed Gold Delta table

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

In [0]:
# Validate the stored Gold table

stored_gold_df = spark.table(gold_table)

stored_gold_row_count = stored_gold_df.count()

print(f"Stored Gold rows: {stored_gold_row_count:,}")

print(
    "Stored count matches Gold DataFrame:",
    stored_gold_row_count == gold_row_count
)

stored_top_products_df = (
    stored_gold_df
    .orderBy(
        col("total_revenue").desc()
    )
    .limit(10)
)

stored_top_products_df.show(10, truncate=False)

Stored Gold rows: 3,057
Stored count matches Gold DataFrame: True
+--------+----------+----------------------------------+-------------------+-------------+----------------------+-----------------------+
|batch_id|stock_code|description                       |total_quantity_sold|total_revenue|distinct_invoice_count|distinct_customer_count|
+--------+----------+----------------------------------+-------------------+-------------+----------------------+-----------------------+
|2009-12 |DOT       |DOTCOM POSTAGE                    |49                 |18574.58     |49                    |0                      |
|2009-12 |85123A    |WHITE HANGING HEART T-LIGHT HOLDER|6406               |17255.35     |304                   |220                    |
|2009-12 |22086     |PAPER CHAIN KIT 50'S CHRISTMAS    |3362               |10169.36     |184                   |111                    |
|2009-12 |15056BL   |EDWARDIAN PARASOL BLACK           |2188               |8671.75      |69              